# Fase 3 — Scripts y mediciones de eficiencia

**Proyecto:** Grupo 3 · MCDI500
**Dataset:** Siniestros de tipo atropello 2023 (CONASET, con datos de Carabineros de Chile)
**Pregunta del proyecto:** ¿qué factores de la vía, del entorno y del contexto del siniestro determinan la probabilidad de que un atropello resulte con víctimas fatales?

En las Fases 1 y 2 trabajamos con los datos. En esta fase revisamos **el código** que los procesa: cómo está ordenado y cuánto se demora. No agregamos datos nuevos ni rehicimos la limpieza; usamos el archivo que dejó listo la Fase 2.

### Cómo medimos

Para cada cálculo escribimos dos versiones (A y B). Primero revisamos que las dos den **el mismo resultado**, porque si una es más rápida pero da otra respuesta, no sirve. Después medimos el tiempo con `timeit` (3 veces, y nos quedamos con el menor, porque a veces el computador está ocupado con otra cosa) y la memoria con `tracemalloc`. Estas funciones están en `src/medicion.py`, así todas las mediciones se hacen igual.

### Relación con el foro técnico

En el foro conversamos sobre cuándo conviene usar recursión. Lo que aplicamos acá es que la recursión sirve cuando **no se sabe cuántos niveles tiene el problema**, y que la memoria que usa depende de esos niveles y no de la cantidad de datos. Por eso solo la usamos en la Medición 1 (las carpetas del proyecto) y no en la tabla de atropellos. También nos quedó la idea de medir antes de decir que algo es mejor.

## 1. Configuración
Importamos nuestros módulos de `src/`: uno para medir y uno por cada medición.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.append("src")
import medicion
import recorrido
import hora
import busqueda

mediciones = []          # se acumula aquí el resultado de cada comparación
print("Python", sys.version.split()[0], "· pandas", pd.__version__)

## 2. Carga del conjunto de la Fase 2
Cargamos el archivo que generó la Fase 2. Los `assert` revisan que sea el archivo correcto antes de medir.

In [ ]:
ruta_procesado = Path("../F1/data/processed/atropellos_2023_features.csv")
df = pd.read_csv(ruta_procesado)

print("Filas:", df.shape[0], "· columnas:", df.shape[1])
assert df.shape[0] == 5541, "El conjunto no tiene las 5.541 filas esperadas"
assert df["Es_Fatal"].sum() == 407, "La variable objetivo no tiene los 407 casos fatales esperados"
print("Casos fatales:", df["Es_Fatal"].sum(), f"({df['Es_Fatal'].mean():.2%})")

## 3. Medición 1 · Recorrido de las carpetas del proyecto

**¿Por qué recursión acá?** La tabla de atropellos no la necesita: tiene región → provincia → comuna, pero siempre son 3 niveles, y eso se resuelve con un `groupby`. Las **carpetas del proyecto** son distintas, porque pueden tener cualquier cantidad de niveles y eso cambia cada vez que creamos una carpeta nueva.

- **A:** `rglob` revisa todo el proyecto, incluida la carpeta `.venv`, y al final descarta lo que no queremos (así estaba en `F1_Definicion.ipynb`).
- **B:** una función recursiva que se salta las carpetas que no queremos sin entrar en ellas.

In [ ]:
raiz = recorrido.encontrar_raiz()
total = sum(1 for _ in raiz.rglob("*"))
en_venv = sum(1 for _ in (raiz / ".venv").rglob("*")) if (raiz / ".venv").exists() else 0
print(f"Rutas en el repositorio: {total:,} · dentro de .venv: {en_venv:,} ({en_venv / total:.1%})\n")

fila = medicion.comparar(
    "rglob + filtro",       lambda: recorrido.arbol_rglob(raiz),
    "recursiva con poda",   lambda: recorrido.arbol_recursivo(raiz),
)
fila["medicion"] = "1 · Recorrido de carpetas"
mediciones.append(fila)

arbol = recorrido.arbol_recursivo(raiz)
print(f"\nElementos mostrados: {len(arbol)} · profundidad: {recorrido.profundidad(arbol)} niveles")

**Interpretación.** La versión A revisa todos los archivos del proyecto aunque casi todos están en `.venv` y después se descartan. La versión B solo revisa lo que se muestra. Como el proyecto tiene pocos niveles de carpetas, la función recursiva se llama pocas veces seguidas, así que no hay riesgo de pasar el límite de Python. **Nos quedamos con la versión B.**

## 4. Medición 2 · Hora del siniestro

**Error que encontramos en la Fase 2.** `Hora_limpia` se sacó de la columna `Hora`, pero esa columna tiene el mismo valor en todas las filas (`1899/12/30 00:00`, un error típico de Excel). Por eso `Hora_limpia` quedó en 0 para todos los atropellos. La hora real está en `Hora_texto`:

In [ ]:
print("Valores distintos de Hora_limpia (F2):", df["Hora_limpia"].unique())
print("Ejemplos de Hora_texto:", df["Hora_texto"].head(3).tolist())

- **A:** recorrer `Hora_texto` fila por fila con `datetime.strptime`.
- **B:** convertir toda la columna de una vez con `pd.to_datetime(...).dt.hour`.

In [ ]:
fila = medicion.comparar(
    "bucle con strptime",   lambda: hora.hora_bucle(df),
    "pd.to_datetime",       lambda: hora.hora_vectorizada(df),
)
fila["medicion"] = "2 · Hora del siniestro"
mediciones.append(fila)

df["Hora_corregida"] = hora.hora_vectorizada(df)
assert df["Hora_corregida"].between(0, 23).all(), "Hay horas fuera del rango 0-23"
print("\nHoras distintas tras la corrección:", df["Hora_corregida"].nunique())
print(df["Hora_corregida"].value_counts().sort_index().to_string())

**Interpretación.** Las dos versiones dan la misma hora para las 5.541 filas. La versión B es más rápida porque pandas convierte la columna completa de una vez, en vez de hacerlo fila por fila. En la memoria pasa algo interesante: la versión B arma la columna de fechas completa antes de sacar la hora, así que ser más rápida no significa necesariamente usar menos memoria.

Con la hora corregida, la variable deja de ser siempre 0 y muestra cómo se reparten los atropellos durante el día.

## 5. Medición 3 · Atropellos fatales por comuna

Para responder la pregunta del proyecto tenemos que consultar cifras por comuna muchas veces.

- **A:** para cada comuna, filtrar la tabla completa y sumar `Es_Fatal`.
- **B:** agrupar una sola vez con `groupby`, guardar el resultado en un diccionario y consultarlo.

In [ ]:
comunas = df["COMUNA1"].unique()
print("Comunas en el conjunto:", len(comunas), "\n")

fila = medicion.comparar(
    "filtrar cada vez",     lambda: busqueda.fatales_filtrando(df, comunas),
    "agrupar una vez",      lambda: busqueda.fatales_agrupando(df, comunas),
)
fila["medicion"] = "3 · Fatales por comuna"
mediciones.append(fila)

fatales = busqueda.fatales_agrupando(df, comunas)
assert sum(fatales.values()) == df["Es_Fatal"].sum(), "La suma por comuna no coincide con el total"
print("\nComunas con más atropellos fatales:")
print(pd.Series(fatales).sort_values(ascending=False).head(5).to_string())

**Interpretación.** La versión A recorre la tabla completa una vez por cada comuna, o sea 284 veces. La versión B la recorre una sola vez y después solo consulta el diccionario, que es casi instantáneo. Fue la diferencia más grande de las tres mediciones. **Nos quedamos con la versión B.**

## 6. Resumen de las mediciones

In [ ]:
tabla = pd.DataFrame(mediciones)[
    ["medicion", "version_a", "tiempo_a_s", "memoria_a_mb",
     "version_b", "tiempo_b_s", "memoria_b_mb", "veces_mas_rapida"]
]
tabla.round(4)

## 7. Conclusiones

En las tres mediciones pasó lo mismo: la versión lenta repetía trabajo (fila por fila, comuna por comuna o archivo por archivo) cuando se podía hacer una sola vez o saltarse desde el principio. Siempre comprobamos que las dos versiones dieran el mismo resultado, así que las versiones rápidas no cambian ninguna cifra del proyecto.

Además, revisar el código con calma nos ayudó a encontrar el error de `Hora_limpia`, que no se notaba solo ejecutando el notebook de la Fase 2.